In [1]:
import pandas as pd
import numpy as np
# Load the dataset, replacing #N/B with NaN
df = pd.read_csv("Education_in_General.csv", na_values=["#N/B"])

In [2]:
df.head()

,ISO_Code,Country,Year,"School life expectancy, primary to tertiary, male (years)","School life expectancy, primary to tertiary, female (years)","Government expenditure on primary education, US$ (millions)","Government expenditure on secondary education, US$ (millions)","Government expenditure on tertiary education, US$ (millions)",Government expenditure on primary education as a percentage of GDP (%),Government expenditure on secondary education as a percentage of GDP (%),Government expenditure on tertiary education as a percentage of GDP (%),Unnamed: 11,Unnamed: 12
0,DZA,Algeria,2010,13.88,14.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AGO,Angola,2010,10.42,8.10,NaN,NaN,NaN,2.08,0.53,0.40,NaN,NaN
2,BWA,Botswana,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BFA,Burkina Faso,2010,7.05,5.97,213.65,63.74,66.71,2.11,0.63,0.66,NaN,NaN
4,CPV,Cabo Verde,2010,11.76,12.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Drop unnamed/empty columns
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

In [4]:
# Display variable types
print(df.dtypes)
print("\nShape:", df.shape)

ISO_Code                                                                     object
Country                                                                      object
Year                                                                          int64
School life expectancy, primary to tertiary, male (years)                   float64
School life expectancy, primary to tertiary, female (years)                 float64
Government expenditure on primary education, US$ (millions)                 float64
Government expenditure on secondary education, US$ (millions)               float64
Government expenditure on tertiary education, US$ (millions)                float64
Government expenditure on primary education as a percentage of GDP (%)      float64
Government expenditure on secondary education as a percentage of GDP (%)    float64
Government expenditure on tertiary education as a percentage of GDP (%)     float64
dtype: object

Shape: (756, 11)


In [5]:
# Count and percentage of missing values per column
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Summary table
missing_df = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing (%)": missing_pct
})

print(missing_df[missing_df["Missing Count"] > 0])

                                                    Missing Count  Missing (%)
School life expectancy, primary to tertiary, ma...            496        65.61
School life expectancy, primary to tertiary, fe...            496        65.61
Government expenditure on primary education, US...            504        66.67
Government expenditure on secondary education, ...            500        66.14
Government expenditure on tertiary education, U...            509        67.33
Government expenditure on primary education as ...            450        59.52
Government expenditure on secondary education a...            443        58.60
Government expenditure on tertiary education as...            448        59.26


In [6]:
# Select only numerical columns for outlier detection
num_cols = df.select_dtypes(include="number").columns

outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Count outliers outside the IQR bounds
    n_outliers = df[(df[col] < lower) | (df[col] > upper)][col].count()
    outlier_summary[col] = {
        "Q1": round(Q1, 2),
        "Q3": round(Q3, 2),
        "IQR": round(IQR, 2),
        "Lower bound": round(lower, 2),
        "Upper bound": round(upper, 2),
        "Outliers count": n_outliers
    }

# Display as DataFrame
outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df)

                                                         Q1       Q3     IQR  \
Year                                                2013.00  2020.00    7.00   
School life expectancy, primary to tertiary, ma...     8.98    12.64    3.66   
School life expectancy, primary to tertiary, fe...     8.03    12.25    4.22   
Government expenditure on primary education, US...    73.90   414.40  340.50   
Government expenditure on secondary education, ...    38.60   375.90  337.30   
Government expenditure on tertiary education, U...    36.64   243.52  206.88   
Government expenditure on primary education as ...     0.98     2.01    1.03   
Government expenditure on secondary education a...     0.67     1.67    1.00   
Government expenditure on tertiary education as...     0.36     0.92    0.57   

                                                    Lower bound  Upper bound  \
Year                                                    2002.50      2030.50   
School life expectancy, primary to tert

In [7]:
# --- CORRECTION 1 : #N/B already handled at load time (na_values=["#N/B"]) ---
# No row deletion : missing data is structural, not erroneous

# Confirm NaN are properly encoded (no residual #N/B strings)
residual = df.isin(["#N/B"]).sum().sum()
print("Residual #N/B strings:", residual)  # Expected : 0

# --- CORRECTION 2 : Cap outliers on GDP % columns only ---
# Outliers in absolute USD are real (large economies) → keep as-is
# GDP % columns : aberrant values above IQR upper bound are capped

gdp_cols = [
    "Government expenditure on primary education as a percentage of GDP (%)",
    "Government expenditure on secondary education as a percentage of GDP (%)",
    "Government expenditure on tertiary education as a percentage of GDP (%)"
]

for col in gdp_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    
    # Cap values above upper bound (floor not needed : no negative GDP %)
    df[col] = df[col].clip(upper=upper)

print("\nCorrection applied : GDP % columns capped at IQR upper bound")
print(df[gdp_cols].describe().round(2))

Residual #N/B strings: 0

Correction applied : GDP % columns capped at IQR upper bound
       Government expenditure on primary education as a percentage of GDP (%)  \
count                                             306.00                        
mean                                                1.55                        
std                                                 0.73                        
min                                                 0.01                        
25%                                                 0.98                        
50%                                                 1.44                        
75%                                                 2.01                        
max                                                 3.55                        

       Government expenditure on secondary education as a percentage of GDP (%)  \
count                                             313.00                          
mean             

In [9]:
# ── CONVERSION 1 : Year int → datetime 
# Required for time series in Power BI (line charts, slicers by year)
df["Year"] = pd.to_datetime(df["Year"], format="%Y")

# ── CONVERSION 2 : Round float columns to 2 decimal places
# Ensures clean DECIMAL(5,2) storage in MySQL and readable KPIs in Power BI

float_cols = df.select_dtypes(include="float64").columns

df[float_cols] = df[float_cols].round(2)

# ── CONVERSION 3 : ISO_Code and Country → explicit string (object → str)
# Avoids silent cast issues when exporting to MySQL VARCHAR
df["ISO_Code"] = df["ISO_Code"].astype(str)
df["Country"]  = df["Country"].astype(str)

# ── VERIFICATION 
print(df.dtypes)
print("\nSample :")
print(df.head(3))

ISO_Code                                                                            object
Country                                                                             object
Year                                                                        datetime64[ns]
School life expectancy, primary to tertiary, male (years)                          float64
School life expectancy, primary to tertiary, female (years)                        float64
Government expenditure on primary education, US$ (millions)                        float64
Government expenditure on secondary education, US$ (millions)                      float64
Government expenditure on tertiary education, US$ (millions)                       float64
Government expenditure on primary education as a percentage of GDP (%)             float64
Government expenditure on secondary education as a percentage of GDP (%)           float64
Government expenditure on tertiary education as a percentage of GDP (%)            float64

In [10]:
# Rename all columns with business-friendly snake_case names
df = df.rename(columns={
    "ISO_Code":                                                                        "country_code",
    "Country":                                                                         "country",
    "Year":                                                                            "year",
    "School life expectancy, primary to tertiary, male (years)":                       "school_life_expectancy_male",
    "School life expectancy, primary to tertiary, female (years)":                     "school_life_expectancy_female",
    "Government expenditure on primary education, US$ (millions)":                     "investment_primary_usd_million",
    "Government expenditure on secondary education, US$ (millions)":                   "investment_secondary_usd_million",
    "Government expenditure on tertiary education, US$ (millions)":                    "investment_tertiary_usd_million",
    "Government expenditure on primary education as a percentage of GDP (%)":          "investment_primary_pct_gdp",
    "Government expenditure on secondary education as a percentage of GDP (%)":        "investment_secondary_pct_gdp",
    "Government expenditure on tertiary education as a percentage of GDP (%)":         "investment_tertiary_pct_gdp"
})

# Verify renaming
print(df.columns.tolist())
print("\nSample :")
print(df.head(3))

['country_code', 'country', 'year', 'school_life_expectancy_male', 'school_life_expectancy_female', 'investment_primary_usd_million', 'investment_secondary_usd_million', 'investment_tertiary_usd_million', 'investment_primary_pct_gdp', 'investment_secondary_pct_gdp', 'investment_tertiary_pct_gdp']

Sample :
  country_code   country       year  school_life_expectancy_male  \
0          DZA   Algeria 2010-01-01                        13.88   
1          AGO    Angola 2010-01-01                        10.42   
2          BWA  Botswana 2010-01-01                          NaN   

   school_life_expectancy_female  investment_primary_usd_million  \
0                          14.65                             NaN   
1                           8.10                             NaN   
2                            NaN                             NaN   

   investment_secondary_usd_million  investment_tertiary_usd_million  \
0                               NaN                              NaN   
1 

In [11]:
# ── FEATURE 1 : Average school life expectancy (male + female)
# Mean of both genders → global schooling KPI per country/year
df["school_life_expectancy_avg"] = df[["school_life_expectancy_male",
                                        "school_life_expectancy_female"]].mean(axis=1).round(2)

# ── FEATURE 2 : Gender gap in schooling (male - female)
# Positive = boys schooled longer / Negative = girls schooled longer
df["gender_gap_schooling"] = (df["school_life_expectancy_male"]
                               - df["school_life_expectancy_female"]).round(2)

# ── FEATURE 3 : Gender parity status
# Binary label based on absolute gender gap threshold of 0.5 years
df["gender_parity_status"] = df["gender_gap_schooling"].apply(
    lambda x: "Parité atteinte" if pd.notna(x) and abs(x) < 0.5 else (
              "Écart significatif" if pd.notna(x) else np.nan)
)

# ── FEATURE 4 : Total education investment in USD (all levels)
# Sum of primary + secondary + tertiary → total public investment volume
df["investment_total_usd_million"] = df[["investment_primary_usd_million",
                                          "investment_secondary_usd_million",
                                          "investment_tertiary_usd_million"]].sum(axis=1, min_count=1).round(2)

# ── FEATURE 5 : Total education investment as % of GDP (all levels)
# Sum of 3 GDP % columns → overall budgetary effort indicator
df["investment_total_pct_gdp"] = df[["investment_primary_pct_gdp",
                                      "investment_secondary_pct_gdp",
                                      "investment_tertiary_pct_gdp"]].sum(axis=1, min_count=1).round(2)

# ── FEATURE 6 : Investment level category (Low / Medium / High)
# Segmentation based on IQR of total GDP % → decision-ready labels
q1 = df["investment_total_pct_gdp"].quantile(0.25)
q3 = df["investment_total_pct_gdp"].quantile(0.75)

df["investment_level_category"] = df["investment_total_pct_gdp"].apply(
    lambda x: "Faible"  if pd.notna(x) and x <= q1 else (
              "Élevé"   if pd.notna(x) and x >= q3 else (
              "Moyen"   if pd.notna(x) else np.nan))
)

# ── FEATURE 7 : Schooling return per USD invested
# Ratio : avg school years / total USD invested → educational efficiency score
df["schooling_return_per_usd"] = (
    df["school_life_expectancy_avg"] / df["investment_total_usd_million"]
).replace([np.inf, -np.inf], np.nan).round(4)

# ── FEATURE 8 : Dominant investment level
# Identifies which education level receives the most public funding
usd_cols = {
    "investment_primary_usd_million":   "Primaire",
    "investment_secondary_usd_million": "Secondaire",
    "investment_tertiary_usd_million":  "Tertiaire"
}

def dominant_level(row):
    # Return NaN if all three values are missing
    values = {k: row[k] for k in usd_cols if pd.notna(row[k])}
    if not values:
        return np.nan
    return usd_cols[max(values, key=values.get)]

df["dominant_investment_level"] = df.apply(dominant_level, axis=1)

# ── FEATURE 9 : Period segmentation
# Groups years into 2 decision phases for comparative temporal analysis
df["period"] = df["year"].dt.year.apply(
    lambda y: "Phase 1 (2010-2015)" if y <= 2015 else "Phase 2 (2016-2023)"
)

# ── VERIFICATION
new_features = [
    "school_life_expectancy_avg",
    "gender_gap_schooling",
    "gender_parity_status",
    "investment_total_usd_million",
    "investment_total_pct_gdp",
    "investment_level_category",
    "schooling_return_per_usd",
    "dominant_investment_level",
    "period"
]

print("New features created :", len(new_features))
print(df[new_features].head(5))
print("\nData types :")
print(df[new_features].dtypes)
print("\nMissing values :")
print(df[new_features].isnull().sum())

New features created : 9
   school_life_expectancy_avg  gender_gap_schooling gender_parity_status  \
0                       14.26                 -0.77   Écart significatif   
1                        9.26                  2.32   Écart significatif   
2                         NaN                   NaN                  NaN   
3                        6.51                  1.08   Écart significatif   
4                       12.36                 -1.19   Écart significatif   

   investment_total_usd_million  investment_total_pct_gdp  \
0                           NaN                       NaN   
1                           NaN                      3.01   
2                           NaN                       NaN   
3                         344.1                      3.40   
4                           NaN                       NaN   

  investment_level_category  schooling_return_per_usd  \
0                       NaN                       NaN   
1                     Moyen          

In [12]:
df.head()

,country_code,country,year,school_life_expectancy_male,school_life_expectancy_female,investment_primary_usd_million,investment_secondary_usd_million,investment_tertiary_usd_million,investment_primary_pct_gdp,investment_secondary_pct_gdp,investment_tertiary_pct_gdp,school_life_expectancy_avg,gender_gap_schooling,gender_parity_status,investment_total_usd_million,investment_total_pct_gdp,investment_level_category,schooling_return_per_usd,dominant_investment_level,period
0,DZA,Algeria,2010-01-01,13.88,14.65,NaN,NaN,NaN,NaN,NaN,NaN,14.26,-0.77,Écart significatif,NaN,NaN,NaN,NaN,NaN,Phase 1 (2010-2015)
1,AGO,Angola,2010-01-01,10.42,8.10,NaN,NaN,NaN,2.08,0.53,0.40,9.26,2.32,Écart significatif,NaN,3.01,Moyen,NaN,NaN,Phase 1 (2010-2015)
2,BWA,Botswana,2010-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Phase 1 (2010-2015)
3,BFA,Burkina Faso,2010-01-01,7.05,5.97,213.65,63.74,66.71,2.11,0.63,0.66,6.51,1.08,Écart significatif,344.1,3.40,Moyen,0.0189,Primaire,Phase 1 (2010-2015)
4,CPV,Cabo Verde,2010-01-01,11.76,12.95,NaN,NaN,NaN,NaN,NaN,NaN,12.36,-1.19,Écart significatif,NaN,NaN,NaN,NaN,NaN,Phase 1 (2010-2015)


In [13]:
df.to_excel("Education_in_General.xlsx",index=False,sheet_name="Education_in_Africa")

In [13]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [14]:
engine.connect()

In [15]:
df.to_sql(
    name="education_in_africa",
    con=engine,
    if_exists="replace",
    index=False
)

756